# Tutorial 5: Transfer Learning / Multi-Fidelity Emulation

When high-fidelity simulations are expensive, we can train on many **cheap
low-fidelity** runs and a few **expensive high-fidelity** runs using the
linear multi-fidelity AR1 model:

$$f_{\rm high}(x) = \rho \cdot f_{\rm low}(x) + \delta(x)$$

where $\rho$ is the scaling factor and $\delta$ is an independent GP
modelling the discrepancy.

**Requirements:** `pip install hic-bayes[multifidelity]` (emukit + GPy)

This notebook compares:
- Standard GP trained on all available low-fidelity points only
- Multi-fidelity GP trained on (all low + few high)-fidelity points

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hic_bayes import fit_pca, Emulator
from hic_bayes.emulator_mf import MultiFidelityEmulator
from hic_bayes.plots import plot_emulator_validation

# Load toy data (both fidelity levels created by generate_toy_data.py)
X       = pd.read_csv('../data/toy/design_points.csv').values
Y_low   = pd.read_csv('../data/toy/simulation_low.csv').values   # cheap sim
Y_high  = pd.read_csv('../data/toy/simulation_high.csv').values  # expensive sim
X_val   = pd.read_csv('../data/toy/validation_design.csv').values
Y_val   = pd.read_csv('../data/toy/validation_simulation.csv').values

print(f'Low fidelity:  X={X.shape}, Y_low={Y_low.shape}')
print(f'High fidelity: X={X.shape}, Y_high={Y_high.shape}')

# Simulate the TL use case: only n_high points for high-fidelity
N_HIGH = 10   # try 5, 10, 20 to see the TL benefit
rng = np.random.default_rng(42)
idx_high = rng.choice(len(X), size=N_HIGH, replace=False)
X_high_sub = X[idx_high]
Y_high_sub = Y_high[idx_high]
print(f'\nUsing {N_HIGH} high-fidelity training points.')

In [ ]:
# PCA fitted on low-fidelity data; both fidelities projected onto same basis
pca = fit_pca(Y_low, n_components=5)
Z_low  = pca.transform(Y_low)
Z_high = pca.transform(Y_high_sub)

param_ranges = np.column_stack([X.min(axis=0), X.max(axis=0)])

In [ ]:
# --- Baseline: standard GP trained only on high-fidelity points ---
print(f'Training standard GP on {N_HIGH} high-fidelity points ...')
em_std = Emulator(n_restarts=2)
em_std.fit(X_high_sub, Z_high, param_ranges, n_jobs=-1)

Z_val_std, _ = em_std.predict(X_val, return_std=False)
Y_val_std = pca.inverse_transform(Z_val_std)

ss_res = np.sum((Y_val - Y_val_std)**2, axis=0)
ss_tot = np.sum((Y_val - Y_val.mean(axis=0))**2, axis=0)
r2_std = (1 - ss_res / ss_tot).mean()
print(f'Standard GP  mean R² = {r2_std:.4f}')

In [ ]:
# --- Multi-fidelity GP ---
print(f'Training multi-fidelity GP (all {len(X)} low + {N_HIGH} high) ...')
try:
    em_mf = MultiFidelityEmulator(n_optimization_restarts=3)
    em_mf.fit(X, Z_low, X_high_sub, Z_high)

    Z_val_mf, _ = em_mf.predict(X_val, return_std=False)
    Y_val_mf = pca.inverse_transform(Z_val_mf)

    ss_res_mf = np.sum((Y_val - Y_val_mf)**2, axis=0)
    r2_mf = (1 - ss_res_mf / ss_tot).mean()
    print(f'Multi-fidelity GP mean R² = {r2_mf:.4f}')
    print(f'\nR² improvement from TL:  {r2_mf - r2_std:+.4f}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    plot_emulator_validation(Y_val, Y_val_std, n_cols=Y_val.shape[1]//2)
    plt.suptitle(f'Standard GP (R²={r2_std:.3f})')
    plt.tight_layout(); plt.show()
    plot_emulator_validation(Y_val, Y_val_mf, n_cols=Y_val.shape[1]//2)
    plt.suptitle(f'Multi-fidelity GP (R²={r2_mf:.3f})')
    plt.tight_layout(); plt.show()

except ImportError as e:
    print(f'\nSkipping multi-fidelity emulator: {e}')
    print('Install with: pip install hic-bayes[multifidelity]')

## Interpretation

- When `N_HIGH` is small, the multi-fidelity GP should significantly outperform
  the standard GP by leveraging the abundant low-fidelity data.
- As `N_HIGH` increases, the gap narrows — the standard GP eventually catches up.
- The break-even point depends on the correlation $\rho$ between fidelity levels
  and the discrepancy complexity.

> **Next step:** use either emulator as a drop-in replacement in `run_analysis.py`
> or `run_tl_analysis.py` to get full posterior distributions.